## How Memory Works in `ConversationalRetrievalChain`

`ConversationBufferMemory` allows the chatbot to remember previous questions and answers. This enables the chatbot to understand follow-up questions and have a natural conversation.

---

#### Step 1: Create the Memory

```python
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)
```

This creates an empty conversation history.

Initially:

```text
chat_history = []
```

---

#### Step 2: Attach the Memory to the Chain

```python
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    memory=memory,
    combine_docs_chain_kwargs={"prompt": PROMPT}
)
```

By passing the `memory` object to the chain, LangChain automatically stores every conversation.

You do **not** need to manually save or retrieve previous messages.

---

#### Step 3: User Asks the First Question

**User**

```text
What does Snapy AI do?
```

Internally, the chain receives:

```text
Question:
"What does Snapy AI do?"

Chat History:
[]
```

The chatbot generates an answer.

Example:

```text
Snapy AI provides AI-powered automation solutions...
```

After generating the answer, the memory is automatically updated.

Memory now contains:

```text
Human:
What does Snapy AI do?

AI:
Snapy AI provides AI-powered automation solutions...
```

---

#### Step 4: User Asks a Follow-up Question

**User**

```text
Who is it designed for?
```

Notice that the user does not say:

```text
Who is Snapy AI designed for?
```

The chatbot still understands what **"it"** refers to because the previous conversation is stored in memory.

Internally, LangChain sends something similar to:

```text
Chat History

Human:
What does Snapy AI do?

AI:
Snapy AI provides AI-powered automation solutions...

Current Questions:
Who is it designed for?
```

The LLM understands that **"it"** refers to **Snapy AI**.

---

#### Step 5: Another Follow-up Question

**User**

```text
Does it have pricing?
```

Internally, the chain now sends:

```text
Chat History

Human:
What does Snapy AI do?

AI:
...

Human:
Who is it designed for?

AI:
...

Current Question:
Does it have pricing?
```

Again, the LLM knows that **"it"** refers to **Snapy AI**.

---

### What Happens Without Memory?

If the memory is removed:

```python
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever()
)
```

Conversation:

```text
User: What does Snapy AI do?

AI: Snapy AI is an AI platform...
```

Next question:

```text
User: Who is it designed for?
```

Internally, the chain only receives:

```text
Question:
Who is it designed for?
```

Since there is no conversation history, the chatbot does not know what **"it"** refers to.

---

#### What Does `ConversationBufferMemory` Store?

It stores the complete conversation in chronological order.

Example:

```text
Human:
What does Snapy AI do?

AI:
Snapy AI is an AI automation platform.

Human:
Is it free?

AI:
It offers both free and paid plans.

Human:
Does it support APIs?

AI:
Yes, it provides API integration.
```

---

#### How Is Memory Updated?

Every time the following code executes:

```python
response = qa_chain({"question": query})
```

LangChain automatically performs these steps:

1. Reads the existing conversation history.
2. Retrieves relevant documents from the vector database.
3. Sends both the retrieved documents and the conversation history to the LLM.
4. Generates an answer.
5. Stores the latest question and answer back into memory.

Workflow:

```text
User Question
      │
      ▼
Read Chat History
      │
      ▼
Retrieve Relevant Documents
      │
      ▼
Send Context + History + Question to LLM
      │
      ▼
Generate Answer
      │
      ▼
Store Question and Answer in Memory
```

---

#### Difference Between Retriever and Memory

These two components serve different purposes.

| Component | Purpose |
|-----------|---------|
| **FAISS Retriever** | Searches the website content to find relevant information. |
| **ConversationBufferMemory** | Stores the conversation between the user and the chatbot. |
| **LLM (GPT)** | Combines the retrieved information and conversation history to generate the final answer. |

Example:

**Website Content**

```text
Snapy AI is an AI automation platform.
```

**Conversation**

```text
User: What does Snapy AI do?

AI: It automates business workflows.

User: Does it integrate with Slack?
```

The **FAISS Retriever** searches the website to find information about Slack integration.

The **ConversationBufferMemory** tells the LLM that **"it"** refers to **Snapy AI**.

The LLM combines both pieces of information to generate a context-aware answer.

---

#### Summary

- `ConversationBufferMemory` remembers the conversation.
- `FAISS Retriever` searches the knowledge base.
- The LLM uses **both** the retrieved documents and the conversation history to generate accurate answers.
- Memory enables the chatbot to understand follow-up questions without requiring the user to repeat the full context.

In [ ]:
import os                          # Used to read environment variables
import bs4                         # BeautifulSoup for parsing HTML
import requests                    # Used to download web pages

# Splits large text into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# OpenAI LLM and embedding model
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# FAISS vector database
from langchain_community.vectorstores import FAISS

# Used to create custom prompts
from langchain_classic.prompts import PromptTemplate

# (Not used in this program)
from langchain_community.document_loaders import WebBaseLoader

# (Not used in this program)
from langchain_community.chat_message_histories import ChatMessageHistory

# Stores conversation history
from langchain_classic.memory import ConversationBufferMemory

In [ ]:
# -------------------------
# Configuration
# -------------------------

from dotenv import load_dotenv
from openai import OpenAI
import os

# Load environment variables from .env
load_dotenv(dotenv_path="../.env")
api_key = os.getenv("OPENAI_API_KEY")
# api_key

# Maximum characters per text chunk
CHUNK_SIZE = 1000

# Number of overlapping characters between chunks
CHUNK_OVERLAP = 200

# Website to scrape
WEBSITE_URL = "https://www.snapy.ai/"

In [ ]:
# Large Language Model used to generate answers
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.4      # Lower temperature = more consistent answers
)

# Embedding model converts text into vectors
embeddings = OpenAIEmbeddings()

In [ ]:
# -------------------------
# Download Website
# -------------------------

def fetch_website_content(url):
    """Downloads the HTML content of a website."""

    try:
        # Send HTTP GET request
        response = requests.get(url)

        # Raise exception if status code is not 200
        response.raise_for_status()

        # Return HTML
        return response.text

    except requests.RequestException as e:
        print(f"Error fetching the website: {e}")
        return None

# -------------------------
# Extract Text
# -------------------------

def scrape_website(url):
    """Extracts plain text from the website."""

    # Download HTML
    content = fetch_website_content(url)

    if not content:
        return []

    # Parse HTML
    soup = bs4.BeautifulSoup(content, "html.parser")

    # Remove HTML tags and keep only readable text
    text_content = soup.get_text(
        separator="\n",
        strip=True
    )

    # Return as a list
    return [text_content]

# -------------------------
# Process Website
# -------------------------

def process_website(url):
    """Creates a FAISS vector database from website text."""

    # Extract website text
    docs = scrape_website(url)

    if not docs:
        print("No content could be extracted from the website.")
        return None

    # Create text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )

    # Split large text into smaller chunks
    splits = text_splitter.split_text(docs[0])

    if not splits:
        print("No valid text chunks were created.")
        return None

    # Convert chunks into embeddings and store in FAISS
    vectorstore = FAISS.from_texts(
        splits,
        embedding=embeddings
    )

    return vectorstore

In [ ]:
# -------------------------
# Build Vector Database
# -------------------------

# Create FAISS vector store
vectorstore = process_website(WEBSITE_URL)

# Stop program if vector database creation failed
if not vectorstore:
    print("Failed to create vectorstore. Exiting.")
    exit(1)

# -------------------------
# Prompt Template (includes chat_history for memory)
# -------------------------

# Instructions given to the LLM
prompt_template = """
Use the following pieces of context to answer the human's question.

If you don't know the answer,
just say that you don't know.

Don't try to make up an answer.

Chat History:
{chat_history}

Context:
{context}

Human:
{question}

Assistant:
"""

# Create LangChain prompt
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["chat_history", "context", "question"]
)

In [ ]:
# -------------------------
# Conversation Memory
# -------------------------

# Stores previous questions and answers
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Create the retriever (separate from the chain)
retriever = vectorstore.as_retriever()

# -------------------------
# Chat Function with Visible Steps
# -------------------------

def format_chat_history(chat_history):
    """Converts message objects into a readable string."""
    if not chat_history:
        return "No previous conversation."

    lines = []
    for msg in chat_history:
        if hasattr(msg, "type") and hasattr(msg, "content"):
            role = "Human" if msg.type == "human" else "AI"
            lines.append(f"{role}: {msg.content}")
    return "\n".join(lines)


def chatbot(query):
    """Returns an answer, showing all 5 internal steps."""

    print("\n" + "=" * 55)
    print("  INTERNAL PROCESS — Step-by-Step")
    print("=" * 55)

    # --------------------------------------------------
    # STEP 1: Read Chat History from Memory
    # --------------------------------------------------
    print("\n[STEP 1] Reading chat history from memory...")
    chat_history_vars = memory.load_memory_variables({})
    chat_history = chat_history_vars.get("chat_history", [])

    if not chat_history:
        print("        → Chat history is empty (this is the first question)")
    else:
        print(f"        → Chat history contains {len(chat_history)} message(s):")
        print(f"          {format_chat_history(chat_history).replace(chr(10), chr(10)+'          ')}")
    print("        ✓ DONE")

    # --------------------------------------------------
    # STEP 2: Retrieve Relevant Documents from FAISS
    # --------------------------------------------------
    print("\n[STEP 2] Retrieving relevant documents from FAISS vector DB...")
    retrieved_docs = retriever.invoke(query)
    print(f"        → Retrieved {len(retrieved_docs)} document chunk(s)")
    for i, doc in enumerate(retrieved_docs):
        # Show first 100 characters of each chunk
        preview = doc.page_content[:100].replace("\n", " ")
        print(f"          Chunk {i+1}: \"{preview}...\"")
    print("        ✓ DONE")

    # --------------------------------------------------
    # STEP 3: Build Prompt and Send to LLM
    # --------------------------------------------------
    print("\n[STEP 3] Assembling prompt with context + history + question...")
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    chat_history_text = format_chat_history(chat_history)

    formatted_prompt = PROMPT.format(
        chat_history=chat_history_text,
        context=context_text,
        question=query
    )
    # Show a preview of what's being sent
    print(f"        → Prompt structure:")
    print(f"          Chat History:  {'Present (' + str(len(chat_history)) + ' messages)' if chat_history else 'Empty'}")
    print(f"          Context:       {len(retrieved_docs)} chunk(s) from FAISS")
    print(f"          Question:      \"{query}\"")
    print("        ✓ DONE")

    # --------------------------------------------------
    # STEP 4: Generate Answer
    # --------------------------------------------------
    print("\n[STEP 4] Sending to LLM and generating answer...")
    response = llm.invoke(formatted_prompt)
    answer = response.content
    print(f"        → LLM generated: \"{answer[:100]}{'...' if len(answer) > 100 else ''}\"")
    print("        ✓ DONE")

    # --------------------------------------------------
    # STEP 5: Store Question and Answer in Memory
    # --------------------------------------------------
    print("\n[STEP 5] Storing question and answer back into memory...")
    memory.save_context({"input": query}, {"output": answer})
    print(f"        → Memory now contains {len(chat_history) + 2} message(s) total")
    print("        ✓ DONE")

    print("\n" + "=" * 55)
    print("  FINAL ANSWER")
    print("=" * 55)

    return answer

In [ ]:
print("Welcome to the Website Chatbot!")
print(f"This chatbot can answer questions about: {WEBSITE_URL}")
print("Type 'quit' to exit.")

## Print statement examples
print("\n" + "=" * 55)
print("FAISS stores embeddings and the positions of those embeddings")
print("The returned indices are the stored vector positions")
print("The English output comes from your own text mapping, not directly from FAISS")
print("\n" + "=" * 55)


while True:

    # Read user question
    user_input = input("\nYou: ")

    # Exit condition
    if user_input.lower() == "quit":
        print("Thank you for using the Website Chatbot. Goodbye!")
        break

    # Generate answer
    response = chatbot(user_input)

    # Display answer
    print(f"\nChatbot: {response}")